# Memory, retrieval time and model size

**Goal:** Compare observed memory, estimated retrieval cost and exploratory size trends.

Run cells from top to bottom. Default cells work offline; model execution is an explicit opt-in and writes only to ignored `outputs/`.

## 1. Set up paths

Find the repository and load the analysis helpers.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "data/final").is_dir(), "Run from the repository or notebooks folder"
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
import matplotlib.pyplot as plt
from sleepinn_study.io import read_json, read_jsonl, output_directory
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})


## 2. Plot observed GPU memory

These are observed peak allocated bytes, not theoretical parameter storage or nvidia-smi total use. Allocated and reserved memory are distinct. Hardware and sequence lengths differ across runs; preserve those limits in interpretation.

In [ ]:
memory = pd.read_csv(ROOT / "results/analysis/memory/observed_vram_summary.csv")
display(memory)
wide = memory.pivot(index="label", columns="precision", values="peak_allocated_gib")
ax = wide[["NF4", "BF16"]].plot.bar(figsize=(10,4), color=["#327c9d", "#d88735"])
ax.set(ylabel="Observed peak allocated GPU memory (GiB)", xlabel="")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(output_directory("figures") / "observed_vram.png")
plt.show()
display((100 * (1 - wide.NF4 / wide.BF16)).rename("NF4 memory reduction (%)"))

## 3. Report retrieval overhead

No-RAG has no retrieval stage. The selected pipeline’s component-cost estimate is therefore the estimated extra retrieval work. It does not include the added answer-generation time caused by a longer prompt and is not isolated end-to-end latency.

In [ ]:
r = pd.read_csv(ROOT / "results/retrieval/retrieval_benchmark.csv")
selected = r[r.strategy.eq("scope_hybrid_rrf_rerank_jiwar")]
display((selected.groupby(["source_pdf", "item_type"]).estimated_component_cost_seconds.mean() * 1000).rename("estimated retrieval milliseconds"))
print("Mean estimated retrieval overhead (ms):", selected.estimated_component_cost_seconds.mean() * 1000)

## 4. Examine size as an exploratory association

Use known local parameter counts only. Closed-model sizes are not publicly established and are excluded from this regression. Average NF4/BF16 effects within each local model before fitting; eight points cannot isolate architecture from training or model size.

In [ ]:
from scipy.stats import linregress
models = pd.DataFrame(read_json(ROOT / "data/config/models.json"))
models = models[models.provider.eq("huggingface")].copy()
models["parameters_b"] = models.safetensors.map(lambda x: x["total"] / 1e9)
e = pd.read_csv(ROOT / "results/analysis/clinical/paired_rag_effects.csv")
size = e[e.precision.isin(["NF4", "BF16"])].groupby("model").delta_pp.mean().reset_index().merge(models, left_on="model", right_on="model_id", validate="one_to_one")
fit = linregress(size.parameters_b, size.delta_pp)
display(size[["name", "parameters_b", "delta_pp"]])
print("Exploratory slope (pp per billion parameters):", fit.slope, "p:", fit.pvalue)
fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(size.parameters_b, size.delta_pp, color="#327c9d")
for row in size.itertuples(): ax.annotate(row.name, (row.parameters_b, row.delta_pp), fontsize=8, xytext=(4,4), textcoords="offset points")
x = sorted(size.parameters_b)
ax.plot(x, [fit.intercept + fit.slope*v for v in x], color="#d88735")
ax.axhline(0, color="#888", linewidth=.8)
ax.set(xlabel="Local model parameters (billions)", ylabel="Mean RAG improvement across NF4/BF16 (pp)")
plt.tight_layout()
plt.savefig(output_directory("figures") / "model_size_rag.png")
plt.show()